# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the Fair² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is defined via a Croissant schema accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if running in Colab or a local environment
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Date Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values. This helps identify how the data is organized and what entities and attributes are present.

In [ ]:
# List all record sets by @id and their fields
print("Available Record Sets (by @id):")
record_sets = []
for rs in metadata.record_sets:
    print(f"  RecordSet @id: {rs['@id']}, name: {rs.get('name', '')}")
    record_sets.append(rs['@id'])
    print("    Fields:")
    for field in rs.get('fields', []):
        print(f"      Field @id: {field['@id']}, name: {field.get('name', '')}, dataType: {field.get('dataType', '')}")
    print()

# Show the total number of RecordSets
print(f"Total record sets found: {len(record_sets)}")

# If there are no record sets, check the recordSet property directly
if not record_sets and hasattr(metadata, 'recordSet') and getattr(metadata, 'recordSet', None):
    rs = metadata.recordSet
    print(f"  RecordSet @id: {rs['@id']} (likely only one)")
    record_sets.append(rs['@id'])
    print("    Fields:")
    for field in rs.get('fields', []):
        print(f"      Field @id: {field['@id']}, name: {field.get('name', '')}, dataType: {field.get('dataType', '')}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Prepare a list of available record set @ids
# For demonstration, let's assume the main record set (update this @id if needed from the overview above)
main_record_set_id = None
if len(record_sets) > 0:
    main_record_set_id = record_sets[0]
else:
    # Try alternate attribute if record_sets is empty
    try:
        main_record_set_id = metadata.recordSet['@id']
    except Exception:
        main_record_set_id = None
if main_record_set_id is None:
    raise Exception("No record set found in the dataset metadata.")

# You can add more record set @ids here if there are multiple
dataframes = {}

print(f"Loading records from RecordSet: {main_record_set_id}")
records = list(dataset.records(record_set=main_record_set_id))

df = pd.DataFrame(records)
dataframes[main_record_set_id] = df
print("DataFrame columns loaded:")
print(df.columns.tolist())
print("\nSample records:")
display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Adapt the selected fields below to the actual column names from the dataset.

> **Note:** Replace `<numeric_field_id>` and `<group_field_id>` with appropriate column names (or field `@id`s) from the loaded DataFrame, e.g. `'abundance'`, `'MW'`, etc.

In [ ]:
# --- Example EDA (adapt field names to match your data) ---

# Replace these with real field ids/column names from print(df.columns) above.
numeric_field = None
for col in df.columns:
    # Try to select the first float or int field, e.g. 'abundance', 'MW', etc.
    if df[col].dtype in [int, float]:
        numeric_field = col
        break
# If types are object, try converting
if numeric_field is None:
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
            if df[col].dtype in [int, float]:
                numeric_field = col
                break
        except:
            continue

if numeric_field is None:
    print("No numeric field found for analysis. Please check df.columns.")
else:
    threshold = df[numeric_field].mean()  # Or choose a suitable threshold
    print(f"Filtering for records with {numeric_field} > {threshold:.2f}")
    filtered_df = df[df[numeric_field] > threshold]
    print(filtered_df.head())
    
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, norm_col]].head())
    
    # Try to select a categorical/group field
    group_field = None
    for col in df.columns:
        if df[col].dtype == 'object' and col != numeric_field:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped (mean) records by '{group_field}':")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between numeric and categorical fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    # Histogram
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    # If a group_field was found, boxplot by group
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we leveraged the `mlcroissant` library to parse and explore mass spectrometry data from the FAIR^2 dataset, as described in its Croissant schema. We:

- Discovered available record sets and field structure by their `@id`
- Loaded tabular data using the schema reference and inspected field values
- Performed EDA—filtering and normalizing a numeric field, grouping by a categorical attribute
- Visualized the distributions to gain insights for further analysis

**Next steps:**
Depending on your analysis goals, you may want to aggregate protein information, compare sample conditions, or apply machine learning techniques to the clean DataFrame.

*Tip: For your own analysis, always reference data elements by their `@id` wherever possible for reproducibility and to leverage schema semantics!*